In [3]:
!pip install --upgrade google-meridian[colab]

  Using cached google_meridian-1.5.3-py3-none-any.whl.metadata (10 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 843.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 73.8 MB/s eta 0:00:00
Using cached google_meridian-1.5.3-py3-none-any.whl (466 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.1/935.1 kB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 78.6 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4

In [62]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import grangercausalitytests
from itertools import permutations
import tensorflow_probability as tfp

from meridian import constants
from meridian.data import data_frame_input_data_builder
from meridian.model import model, spec
from meridian.analysis import analyzer

In [28]:
from google.colab import files
uploaded = files.upload()

Saving monthly_mocha.csv to monthly_mocha.csv


In [44]:
import io
df = pd.read_csv(io.BytesIO(uploaded['monthly_mocha.csv']))
df = df.drop(columns=["date"], errors="ignore")
df = df.loc[:, (df != 0).any(axis=0)]

df["time"] = np.arange(len(df))

In [31]:
df.head()

,subscriptions,meta_spend,meta_impressions,google_spend,google_impressions,snapchat_spend,snapchat_impressions,tiktok_spend,tiktok_impressions,moloco_spend,moloco_impressions,liveintent_spend,liveintent_impressions,beehiiv_spend,beehiiv_impressions,amazon_spend,amazon_impressions
0,15540,91538.06648,16572258,116667.9945,6473132,94750.04035,3420454,0.0,0,6564.524233,367206,37766.44904,371854,18190.10332,181901,0.0,0
1,14525,93840.18612,25300600,180486.9558,9487127,99447.23218,3235285,0.0,0,18111.083980,820589,38543.27888,347850,20063.91811,200639,0.0,0
2,16880,48403.06780,14099214,200817.3250,7909118,84738.57435,4766750,0.0,0,9714.794608,369806,39697.42202,322865,20828.00074,208280,0.0,0
3,20113,49470.96783,13652072,215770.9242,7789279,83204.40500,4022680,0.0,0,16831.841440,554980,40561.34006,570418,24097.34426,240973,0.0,0
4,16492,48948.28744,10121002,209231.9668,6806878,82642.37271,4532105,0.0,0,17624.908800,894891,40012.42040,483619,19967.60420,199676,0.0,0


In [45]:
import numpy as np
from itertools import permutations
from statsmodels.tsa.stattools import grangercausalitytests

base_df = df.copy()

drop_cols = ["time"]
base_df = base_df.drop(columns=drop_cols, errors="ignore")
base_df = base_df.select_dtypes(include=[np.number])

channel_diff = base_df.diff().dropna()

In [46]:
gc_results = {}
channels = channel_diff.columns

for x1, x2 in permutations(channels, 2):

    try:
        test_result = grangercausalitytests(
            channel_diff[[x1, x2]],
            maxlag=3,
            verbose=False
        )

        p_values = {
            lag: test_result[lag][0]['ssr_ftest'][1]
            for lag in range(1, 4)
        }

        gc_results[(x1, x2)] = p_values

    except Exception:
        continue

gc_df = pd.DataFrame(gc_results).T.rename(columns=lambda x: f"lag_{x}_pvalue")

print(gc_df.head())

                                  lag_1_pvalue  lag_2_pvalue  lag_3_pvalue
subscriptions meta_spend              0.234904      0.557704      0.962831
              meta_impressions        0.611587      0.417959      0.906033
              google_spend            0.986734      0.117264      0.460338
              google_impressions      0.609708      0.255174      0.142342
              snapchat_spend          0.657851      0.163870      0.466197


In [47]:
import numpy as np
import pandas as pd

long_df = (
    gc_df.reset_index()
    .rename(columns={"level_0": "target", "level_1": "driver"})
    .melt(id_vars=["target", "driver"], var_name="lag", value_name="p_value")
)

long_df["lag"] = long_df["lag"].str.extract(r"lag_(\d+)_pvalue").astype(int)

summary_df = (
    long_df.sort_values(["target", "driver", "p_value"])
    .groupby(["target", "driver"], as_index=False)
    .first()
    .rename(columns={"p_value": "min_pvalue", "lag": "best_lag"})
)

def fdr_bh(pvals):
    pvals = np.asarray(pvals)
    order = np.argsort(pvals)
    ranked = pvals[order]
    q = ranked * len(pvals) / (np.arange(1, len(pvals) + 1))
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.clip(q, 0, 1)
    out = np.empty_like(q)
    out[order] = q
    return out

summary_df["fdr_q"] = fdr_bh(summary_df["min_pvalue"].values)

significant = summary_df[summary_df["fdr_q"] < 0.05]

drivers = significant["driver"].unique().tolist()

print("SIGNIFICANT CAUSAL DRIVERS:")
print(drivers)

print("\nTOP RELATIONSHIPS:")
print(summary_df.sort_values("fdr_q").head(10))

SIGNIFICANT CAUSAL DRIVERS:
['meta_impressions', 'meta_spend', 'liveintent_impressions', 'moloco_impressions', 'google_spend', 'snapchat_impressions', 'liveintent_spend', 'amazon_impressions', 'amazon_spend', 'moloco_spend', 'snapchat_spend', 'tiktok_spend']

TOP RELATIONSHIPS:
                     target                  driver  best_lag    min_pvalue  \
170      moloco_impressions            moloco_spend         3  2.941529e-10   
24             amazon_spend              meta_spend         3  8.202508e-09   
8        amazon_impressions              meta_spend         3  7.218414e-09   
23             amazon_spend        meta_impressions         3  3.867227e-07   
7        amazon_impressions        meta_impressions         3  3.122027e-07   
107  liveintent_impressions    snapchat_impressions         1  2.775043e-06   
186            moloco_spend      moloco_impressions         3  1.202111e-05   
102  liveintent_impressions        liveintent_spend         3  6.226513e-05   
75       g

In [48]:
lags = [1, 2, 3]

df_lagged = df.copy()

for col in drivers:
    if col in df_lagged.columns:
        for lag in lags:
            df_lagged[f"{col}_lag{lag}"] = df_lagged[col].shift(lag)

df_lagged = df_lagged.dropna()

print("Final shape:", df_lagged.shape)
print("Lagged features created for:", drivers)

Final shape: (71, 54)
Lagged features created for: ['meta_impressions', 'meta_spend', 'liveintent_impressions', 'moloco_impressions', 'google_spend', 'snapchat_impressions', 'liveintent_spend', 'amazon_impressions', 'amazon_spend', 'moloco_spend', 'snapchat_spend', 'tiktok_spend']


In [55]:
from meridian.data import data_frame_input_data_builder
import pandas as pd
import numpy as np

df = df.copy()

df["time"] = pd.date_range(
    start="2022-01-01",
    periods=len(df),
    freq="MS"
).strftime("%Y-%m-%d")

df["revenue_per_conversion"] = 1.0
df["population"] = 1.0

df_lagged = df.copy()

kpi_col = "subscriptions"

channels = [
    "snapchat",
    "tiktok",
    "moloco",
    "liveintent",
    "beehiiv",
    "amazon"
]

media_cols = [f"{c}_impressions" for c in channels]
media_spend_cols = [f"{c}_spend" for c in channels]

builder = data_frame_input_data_builder.DataFrameInputDataBuilder(
    kpi_type="non_revenue",
    default_kpi_column=kpi_col,
    default_revenue_per_kpi_column="revenue_per_conversion",
)

builder = (
    builder.with_kpi(df_lagged)
    .with_revenue_per_kpi(df_lagged)
    .with_population(df_lagged)
    .with_media(
        df_lagged,
        media_cols=media_cols,
        media_spend_cols=media_spend_cols,
        media_channels=channels
    )
)

data = builder.build()

print("MMM input data built successfully")

MMM input data built successfully


/usr/local/lib/python3.12/dist-packages/meridian/data/input_data_builder.py:715: UserWarning: The `population` argument is ignored in a nationally aggregated model. It will be reset to [1, 1, ..., 1]
  warnings.warn(


In [58]:
roi_mu = 0.2
roi_sigma = 0.9

prior = spec.prior_distribution.PriorDistribution(
    roi_m=tfp.distributions.LogNormal(
        loc=roi_mu,
        scale=roi_sigma,
        name=constants.ROI_M
    )
)

model_spec = spec.ModelSpec(
    prior=prior,
    enable_aks=True
)

mmm = model.Meridian(
    input_data=data,
    model_spec=model_spec
)

print("Model initialized successfully")

Model initialized successfully


/usr/local/lib/python3.12/dist-packages/meridian/model/model.py:75: UserWarning: In a nationally aggregated model, the `media_effects_dist` will be reset to `normal`.
  warnings.warn(


In [71]:
analyzer_obj = analyzer.Analyzer(mmm)

roi = analyzer_obj.roi()
mroi = analyzer_obj.marginal_roi()
inc = analyzer_obj.incremental_outcome()

print(roi)

/tmp/ipykernel_6530/1805317195.py:1: DeprecationWarning: The `meridian` argument is deprecated and will be removed in a future version. Use `model_context` instead.
  analyzer_obj = analyzer.Analyzer(mmm)


NotFittedModelError: sample_posterior() must be called prior to calling this method.